# EDA: synthetic ransomware-activity windows

Run `python -m ml.generate_dataset` first. Each row is one 10-second window; features come from `app/detection/features.py`.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))
import matplotlib.pyplot as plt
import pandas as pd

from app.detection.features import FEATURE_NAMES

df = pd.read_csv("../data/dataset.csv")
df["primary"] = df["scenario"].str.split("+").str[0]
df.shape

## Class balance and scenarios

In [ ]:
df["label"].value_counts(), df.groupby(["label", "primary"]).size()

## Feature summary by class

In [ ]:
df.groupby("label")[list(FEATURE_NAMES)].median().T

## Distributions: where do benign and attack overlap?

Overlap is intentional: hard negatives (browser cache, ML checkpoints, zip+cleanup) and hard positives (slow, partial, in-place attacks).

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, f in zip(axes.ravel(), FEATURE_NAMES):
    for label, name in [(0, "benign"), (1, "attack")]:
        ax.hist(df.loc[df.label == label, f], bins=40, alpha=0.5, label=name, log=True)
    ax.set_title(f)
axes[0, 0].legend()
plt.tight_layout()

## Hardest benign scenarios (high entropy that is *not* an attack)

In [ ]:
cols = [
    "frac_high_entropy",
    "frac_high_entropy_unexpected",
    "unknown_ext_count",
    "mean_entropy_delta",
]
df.groupby("primary")[cols].mean().sort_values("frac_high_entropy_unexpected", ascending=False)

## Feature correlation

In [ ]:
df[list(FEATURE_NAMES)].corr().round(2)